In [1]:
from pathlib import Path
import pandas as pd

csv_path = Path('sensor_log.csv')

sample = pd.DataFrame({
    'time': ['2026-08-24 09:00', '2026-08-24 09:01',
             '2026-08-24 09:02', '2026-08-24 09:03'],
    'sensor_id': ['S1', 'S1', 'S2', 'S2'],
    'temp': [25.0, None, 35.0, 42.0]
})

sample.to_csv(csv_path, index=False)
print('저장 위치:', csv_path.resolve())

저장 위치: C:\Users\User\Documents\git\HINT\AI를 위한 python\sensor_log.csv


In [2]:
def load_log(path):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(
            f'파일이 없습니다: {path.resolve()}'
        )

    df = pd.read_csv(path)

    required = {'time', 'sensor_id', 'temp'}
    missing = required - set(df.columns)

    if missing:
        raise ValueError(f'필수 열 누락: {sorted(missing)}')

    df['time'] = pd.to_datetime(df['time'], errors='coerce')
    df['temp'] = pd.to_numeric(df['temp'], errors='coerce')

    return df

In [3]:
def clean_log(df):
    df = df.drop_duplicates().copy()

    med = df.groupby('sensor_id')['temp'].transform('median')
    df['temp'] = df['temp'].fillna(med)

    overall = df['temp'].median()
    df['temp'] = df['temp'].fillna(overall)

    needed = ['time', 'sensor_id', 'temp']
    df = df.dropna(subset=needed).copy()

    return df

In [4]:
def flag_status(df, warn=30, danger=40):
    df = df.copy()

    def classify(temp):
        if temp >= danger:
            return '위험'
        if temp >= warn:
            return '주의'
        return '정상'

    df['status'] = df['temp'].apply(classify)

    return df

In [5]:
try:
    df = load_log('sensor_log.csv')
    df = clean_log(df)
    df = flag_status(df)
    print(df.to_string(index=False))
except (FileNotFounderror, ValueError) as error:
    print('실행 오류:', error)

               time sensor_id  temp status
2026-08-24 09:00:00        S1  25.0     정상
2026-08-24 09:01:00        S1  25.0     정상
2026-08-24 09:02:00        S2  35.0     주의
2026-08-24 09:03:00        S2  42.0     위험
